# 03 — Pemodelan BiLSTM pada Data Baru (v2): 5 Varian Empiris Multi-Seed
Notebook ini menguji 5 perlakuan penanganan ketimpangan kelas pada BiLSTM dengan evaluasi 3-seeds (`42, 123, 456`):
1. **Natural Baseline**
2. **Class Weight (CW)**
3. **Random Oversampling (ROS)**
4. **Random Undersampling (RUS)**
5. **SMOTE**


In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, recall_score, classification_report
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

if Path.cwd().name == "notebooks":
    os.chdir("..")


## 1. Load Data & Tokenisasi


In [2]:
df = pd.read_csv("Data/processed/banjir_processed_v2.csv")
tr_val, test_df = train_test_split(df, test_size=0.20, stratify=df['label'], random_state=42)
train_df, val_df = train_test_split(tr_val, test_size=0.10, stratify=tr_val['label'], random_state=42)

tok = Tokenizer(num_words=10000, oov_token="<OOV>")
tok.fit_on_texts(train_df['processed_text_v2'])

X_tr = pad_sequences(tok.texts_to_sequences(train_df['processed_text_v2']), maxlen=50, padding='post', truncating='post')
X_va = pad_sequences(tok.texts_to_sequences(val_df['processed_text_v2']), maxlen=50, padding='post', truncating='post')
X_te = pad_sequences(tok.texts_to_sequences(test_df['processed_text_v2']), maxlen=50, padding='post', truncating='post')

y_tr = train_df['label'].values
y_va = val_df['label'].values
y_te = test_df['label'].values


## 2. Definisi Model BiLSTM


In [3]:
def build_bilstm():
    model = Sequential([
        Embedding(10000, 128),
        Bidirectional(LSTM(64)),
        Dropout(0.3),
        Dense(3, activation='softmax')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model


## 3. Pelatihan Multi-Seed (Rata-rata 3 Seeds)


In [4]:
seeds = [42, 123, 456]
results = []

for s in seeds:
    tf.random.set_seed(s)
    np.random.seed(s)
    m = build_bilstm()
    es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    m.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=20, batch_size=16, callbacks=[es], verbose=0)
    preds = np.argmax(m.predict(X_te, verbose=0), axis=1)
    
    acc = accuracy_score(y_te, preds)
    f1 = f1_score(y_te, preds, average='macro')
    rec_net = recall_score(y_te, preds, labels=[1], average='macro')
    results.append({"Seed": s, "Accuracy": acc, "Macro_F1": f1, "Neutral_Recall": rec_net})

res_df = pd.DataFrame(results)
print("=== HASIL BILSTM NATURAL BASELINE (3 SEEDS) ===")
print(res_df)
print(f"\nMean Accuracy: {res_df['Accuracy'].mean()*100:.2f}% (±{res_df['Accuracy'].std()*100:.2f}%)")
print(f"Mean Macro F1: {res_df['Macro_F1'].mean()*100:.2f}% (±{res_df['Macro_F1'].std()*100:.2f}%)")
print(f"Mean Neutral Recall: {res_df['Neutral_Recall'].mean()*100:.2f}%")


=== HASIL BILSTM NATURAL BASELINE (3 SEEDS) ===
   Seed  Accuracy  Macro_F1  Neutral_Recall
0    42  0.712717  0.516839        0.009934
1   123  0.723121  0.637472        0.314570
2   456  0.721965  0.516760        0.000000

Mean Accuracy: 71.93% (±0.57%)
Mean Macro F1: 55.70% (±6.97%)
Mean Neutral Recall: 10.82%
